[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/07_threshold_and_failure_cases.ipynb)

# Bài 07 — Ngưỡng quyết định, xác suất và các trường hợp Logistic Regression thất bại

**🎯 Mục tiêu**
- Ngưỡng 0.5 **không** phải lúc nào cũng đúng — chọn ngưỡng theo chi phí của từng loại sai.
- Xác suất của Logistic Regression có đáng tin không? (giới thiệu calibration)
- 5 failure case: dữ liệu tách được hoàn toàn, ranh giới phi tuyến, mất cân bằng lớp, outlier, feature tương quan mạnh.
- Template model đầy đủ cho Logistic Regression.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_classification_2d, make_customer_churn, make_fraud_transactions
from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ngưỡng quyết định (decision threshold)

Model cho xác suất $p = P(y = 1 \mid x)$; ta dự đoán $y = 1$ khi $p \ge t$. Ngưỡng $t = 0.5$ chỉ tối ưu khi **hai loại sai có chi phí như nhau**. Thực tế hiếm khi như vậy:

| Bài toán | Sai tốn kém hơn | Ngưỡng nên |
|---|---|---|
| Sàng lọc ung thư | bỏ sót người bệnh (false negative) | **thấp** (vd 0.1–0.3) |
| Lọc spam | đánh dấu nhầm email quan trọng là spam (false positive) | **cao** (vd 0.9) |
| Chăm sóc khách hàng sắp churn | tuỳ chi phí ưu đãi vs giá trị khách hàng | tính theo chi phí |

In [ ]:
churn = make_customer_churn(n=6000, seed=2)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]

rows = []
for t in [0.2, 0.3, 0.5, 0.7]:
    pred = p_te >= t
    tp, fp = np.sum(pred & (y_te == 1)), np.sum(pred & (y_te == 0))
    fn = np.sum(~pred & (y_te == 1))
    rows.append({"ngưỡng": t, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "số KH được liên hệ": int(pred.sum())})
pd.DataFrame(rows).round(3)

- **Precision**: trong số khách hàng model cảnh báo, bao nhiêu % thật sự churn?
- **Recall**: trong số khách hàng thật sự churn, model bắt được bao nhiêu %?
- Hạ ngưỡng → recall tăng, precision giảm (và ngược lại). Chi tiết ở chương 14.

### Chọn ngưỡng theo chi phí kinh doanh

Giả sử: gửi ưu đãi giữ chân tốn **50**; giữ được một khách hàng sắp rời đi (thành công 40% số lần) đem lại **600**. Lợi nhuận kỳ vọng theo ngưỡng:

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
profit = [np.sum((p_te >= t) & (y_te == 1)) * 0.4 * 600 - np.sum(p_te >= t) * 50 for t in thresholds]
best_t = thresholds[int(np.argmax(profit))]
plt.plot(thresholds, profit)
plt.axvline(best_t, color="r", ls="--", label=f"ngưỡng tối ưu ≈ {best_t:.2f}")
plt.axvline(0.5, color="gray", ls=":", label="ngưỡng mặc định 0.5")
plt.xlabel("ngưỡng")
plt.ylabel("lợi nhuận kỳ vọng")
plt.legend()
plt.show()

> ⚠️ Chọn ngưỡng trên **validation** (hoặc OOF predictions), không phải trên test.

Khi chi phí và xác suất đều chính xác, ngưỡng tối ưu lý thuyết là $t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$ — điều này đòi hỏi **xác suất phải được calibrate tốt**.

## 2. Xác suất có đáng tin không? (calibration)

Model nói "xác suất 70%" — trong số các trường hợp như vậy, có thật khoảng 70% là dương tính không? Logistic Regression (không regularize quá mạnh, đúng giả định) thường **khá calibrated**; nhiều model khác (SVM, Naive Bayes, Random Forest) thì không. **Reliability diagram**:

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_te, p_te, n_bins=10)
plt.plot([0, 1], [0, 1], "k--", label="calibrate hoàn hảo")
plt.plot(prob_pred, prob_true, "o-", label="Logistic Regression")
plt.xlabel("xác suất dự đoán trung bình")
plt.ylabel("tỉ lệ dương tính thực tế")
plt.legend()
plt.show()

Đường gần đường chéo → xác suất đáng tin. Chương 37 học sâu về calibration (Platt scaling, isotonic).

## 3. Failure case 1 — Dữ liệu tách được hoàn toàn (perfect separation)

Nếu tồn tại siêu phẳng tách **hoàn hảo** hai lớp, cross-entropy có thể giảm mãi bằng cách **phóng to $\|\mathbf w\|$** → sigmoid dốc dần thành hàm bậc thang, trọng số tiến ra **vô cùng**, gradient descent không bao giờ hội tụ, xác suất trở nên cực đoan (0 hoặc 1).

In [ ]:
Xs = np.r_[rng.normal(-2, 0.5, 30), rng.normal(2, 0.5, 30)]     # hai lớp tách rời nhau hoàn toàn
ys = np.r_[np.zeros(30), np.ones(30)]

# Gradient descent KHÔNG regularization: theo dõi w theo số vòng lặp
w, b = 0.0, 0.0
for t in range(1, 100_001):
    p = 1 / (1 + np.exp(-(w * Xs + b)))
    w -= 0.5 * np.mean((p - ys) * Xs)
    b -= 0.5 * np.mean(p - ys)
    if t in (10, 100, 1_000, 10_000, 100_000):
        print(f"sau {t:>7,} vòng lặp: w = {w:5.2f} | P(y=1 | x = 0.3) = {1 / (1 + np.exp(-(w * 0.3 + b))):.3f}")

Trọng số **tăng mãi** (theo log của số vòng lặp) — không bao giờ hội tụ — và xác suất cho một điểm **gần ranh giới** ngày càng cực đoan. Các thư viện dừng lại khi gradient đủ nhỏ, nên kết quả phụ thuộc tiêu chí dừng — một dấu hiệu bài toán "không xác định". **Khắc phục:** regularization giữ $w$ hữu hạn (mặc định `C = 1` của sklearn là một lý do tốt!):

In [ ]:
for C in [0.1, 1.0, 100.0]:
    m = LogisticRegression(C=C).fit(Xs.reshape(-1, 1), ys)
    print(f"C = {C:<6g} w = {m.coef_[0, 0]:5.2f} | P(y=1 | x = 0.3) = {m.predict_proba([[0.3]])[0, 1]:.3f}")

## 4. Failure case 2 — Ranh giới phi tuyến

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, kind in zip(axes, ["moons", "xor"]):
    Xk, yk = make_classification_2d(kind, n=400, noise=0.2)
    lin = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xk, yk)
    plot_decision_boundary(lin, Xk, yk, ax=ax)
    ax.set_title(f"{kind}: accuracy = {lin.score(Xk, yk):.2f} — ranh giới thẳng không đủ")
plt.show()

**Khắc phục:** feature phi tuyến (đa thức — bài 03; tích $x_1x_2$ giải quyết XOR), kernel (SVM — chương 13), model phi tuyến (tree, neural network).

## 5. Failure case 3 — Mất cân bằng lớp

Với 1% gian lận, model có thể đạt 99% accuracy bằng cách **gần như không bao giờ** dự đoán "gian lận":

In [ ]:
fraud = make_fraud_transactions(n=20000, fraud_rate=0.01, seed=1)
Xf = fraud[["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]].astype(float)
Xf["amount_vnd"] = np.log(Xf["amount_vnd"])
yf = fraud["is_fraud"].to_numpy()
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf)
for cw in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=cw, max_iter=2000)).fit(Xf_tr, yf_tr)
    pred = m.predict(Xf_te)
    recall = np.sum(pred & (yf_te == 1)) / yf_te.sum()
    print(f"class_weight = {cw!s:9s} accuracy = {np.mean(pred == yf_te):.3f} | recall gian lận = {recall:.2f} | số cảnh báo = {pred.sum()}")

**Khắc phục:** metric phù hợp (precision/recall, PR-AUC — chương 14), `class_weight="balanced"`, chọn ngưỡng, resampling (chương 19).

## 6. Failure case 4 & 5 — Outlier và feature tương quan mạnh

- **Outlier ở feature** ($x$ rất lớn) có leverage cao, có thể làm lệch ranh giới (ít nghiêm trọng hơn Linear Regression vì sigmoid bão hoà, nhưng vẫn có ảnh hưởng). → Chuẩn hoá bền vững, log-transform, clip.
- **Feature tương quan mạnh** → hệ số không ổn định, khó diễn giải (giống chương 06, bài 08). → Regularization, bỏ feature dư thừa.

## 7. Template model: Logistic Regression

| Phần | Nội dung |
|---|---|
| Problem | phân loại nhị phân (mở rộng: OvR, softmax) |
| Intuition / Geometry | siêu phẳng $\mathbf w^\top\mathbf x + b = 0$; khoảng cách có dấu tới siêu phẳng → xác suất qua sigmoid |
| Probability | $y \mid \mathbf x \sim \text{Bernoulli}(\sigma(\mathbf w^\top\mathbf x + b))$; log-odds tuyến tính |
| Objective | binary cross-entropy (+ L2/L1) |
| Optimization | GD/SGD, L-BFGS, Newton/IRLS; bài toán **lồi** |
| Assumptions | log-odds tuyến tính theo feature; các mẫu độc lập; ít đa cộng tuyến |
| Complexity | train $O(k\,n\,d)$; predict $O(d)$ |
| Hyperparameters | `C` (= 1/λ), `penalty` (l1/l2/elasticnet), `class_weight`, ngưỡng |
| Preprocessing | **cần** scaling (vì regularization + hội tụ); one-hot cho category; impute missing |
| Strengths | nhanh, lồi, xác suất khá calibrated, diễn giải bằng odds ratio, baseline mạnh |
| Weaknesses | ranh giới tuyến tính; perfect separation; nhạy outlier ở feature; cần feature engineering |
| Interpretation | $e^{w_j}$ = odds ratio khi $x_j$ tăng 1 đơn vị |

## 🧠 Kiểm tra nhanh

1. Khi nào nên hạ ngưỡng xuống dưới 0.5? Cho ví dụ.
2. Vì sao dữ liệu tách được hoàn toàn làm trọng số tiến ra vô cùng khi không regularize?
3. Accuracy 99% trên dữ liệu 1% gian lận — có nghĩa model tốt không?
4. $C_{FP} = 1$, $C_{FN} = 9$. Ngưỡng tối ưu lý thuyết là bao nhiêu?

## 📚 Tài liệu tham khảo

- Elkan (2001) — *The Foundations of Cost-Sensitive Learning*
- Niculescu-Mizil & Caruana (2005) — *Predicting Good Probabilities With Supervised Learning*
- ISLR, §4.3 & §4.4